In [1]:
import pandas as pd

In [2]:
df_train = pd.read_parquet("../../data/churn-prediction-25-26/train.parquet")

In [3]:
df_tnos = df_train.groupby(["userId"])["sessionId"].nunique()\
        .reset_index(name="total_number_of_sessions")

In [4]:
df_tnos.sort_values("total_number_of_sessions", ascending=False)

,userId,total_number_of_sessions
2204,1113757,116
10815,1564221,103
18936,1989666,101
18938,1989862,100
14753,1766016,97
...,...,...
5015,1254722,1
232,1010902,1
5019,1254985,1
1192,1060099,1


In [ ]:
# average time per session
df_atps_max = df_train.groupby(["userId", "sessionId"])["time"].max()\
        .reset_index(name="session_time_end")

# average time per session
df_atps_min = df_train.groupby(["userId", "sessionId"])["time"].min()\
        .reset_index(name="session_time_start")

df_atps = df_atps_max.merge(df_atps_min, how = "inner", on = ["userId", "sessionId"])

In [ ]:
df_atps["time_per_session"] = df_atps["session_time_end"] - df_atps["session_time_start"]

In [ ]:
# time delta to previous session of the same user
df_atps["time_since_prev_session"] = (
    df_atps.groupby(["userId"])["session_time_start"].diff()  # Timedelta
)

In [ ]:
temp_df = right=df_train.groupby("userId")["time"].max().\
    reset_index(name="time_last_used")

df_atps = df_atps.merge(right = temp_df, on = "userId", how = "inner")

In [ ]:
import datetime

df_atps["today"] = datetime.date(year = 2018, month = 11, day = 20)

In [ ]:
df_atps

In [ ]:
#df_atps["days_not_used"] = df_atps["today"] - df_atps["time_last_used"]

df_atps["today"] = pd.to_datetime(df_atps["today"])
df_atps["time_last_used"] = pd.to_datetime(df_atps["time_last_used"])

In [ ]:
df_atps["days_not_used"] = df_atps["today"] - df_atps["time_last_used"]

In [ ]:
df_atps

In [ ]:
# Mean time per session
df_avtps = df_atps.groupby(["userId"])["time_per_session"].mean().\
reset_index(name="mean_time_per_session")

In [ ]:
df_avtps

In [ ]:
df_atps

In [ ]:
# mean days between sessions

df_mdbs = df_atps.groupby("userId")["time_since_prev_session"].mean().\
reset_index(name="mean_days_between_sessions")

In [ ]:
df_mdbs

In [ ]:
df_atps

In [ ]:
# days since last session
df_dsls = df_atps[["userId", "days_not_used"]].drop_duplicates()

In [ ]:
df_dsls

In [ ]:
x = df_train[["userId", "sessionId", "time"]].sort_values(["userId", "sessionId", "time"])

In [ ]:
df_atps

In [ ]:
# ensure timestamp column is datetime
df_train["registration"] = pd.to_datetime(df_train["registration"])

In [ ]:
# sort so earlier sessions come first per user
df_train = df_train.sort_values(["userId", "registration"])

In [ ]:
# time delta to previous session of the same user
df_train["time_since_prev_session"] = (
    df_train.groupby(["userId", "sessionId"])["registration"].diff()  # Timedelta
)

In [ ]:
# if you prefer minutes (float)
d = df_train["minutes_since_prev_session"] = df_train["time_since_prev_session"].dt.total_seconds() / 60